# 01_setup — Phase 2 Delta tables

Creates the Bronze, Silver, quarantine, and operations tables. It is safe to rerun.

In [ ]:
# MAGIC %run ./00_config


In [ ]:
for schema in (BRONZE_SCHEMA, SILVER_SCHEMA, OPS_SCHEMA):
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {table_ref(schema)}')

spark.sql(f'''CREATE TABLE IF NOT EXISTS {table_ref(BRONZE_EVENTS)} (
 event_id STRING NOT NULL COMMENT 'Unique play identifier', user_id STRING NOT NULL COMMENT 'Pseudonymous listener', track_id STRING NOT NULL COMMENT 'Spotify track URI', played_at STRING NOT NULL COMMENT 'Source UTC timestamp', ms_played BIGINT, skipped BOOLEAN, device_type STRING, source STRING NOT NULL, _source_file STRING NOT NULL, _partition_dt DATE NOT NULL, _batch_id STRING NOT NULL, _rescued_data STRING, load_timestamp TIMESTAMP NOT NULL COMMENT 'Bronze load time'
) USING DELTA PARTITIONED BY (_partition_dt) COMMENT 'Append-only raw event deliveries' ''')
spark.sql(f'''CREATE TABLE IF NOT EXISTS {table_ref(BRONZE_USERS)} (user_id STRING NOT NULL, country STRING, timezone STRING, is_premium BOOLEAN, _source_file STRING NOT NULL, _batch_id STRING NOT NULL, _rescued_data STRING, load_timestamp TIMESTAMP NOT NULL) USING DELTA COMMENT 'Append-only user snapshots' ''')
spark.sql(f'''CREATE TABLE IF NOT EXISTS {table_ref(BRONZE_CATALOG)} (track_id STRING NOT NULL, track_name STRING, artist STRING, genre STRING, release_year INT, duration_sec INT, popularity INT, danceability DOUBLE, energy DOUBLE, valence DOUBLE, tempo_bpm DOUBLE, _source_file STRING NOT NULL, _batch_id STRING NOT NULL, _rescued_data STRING, load_timestamp TIMESTAMP NOT NULL) USING DELTA COMMENT 'Append-only catalog snapshots' ''')
spark.sql(f'''CREATE TABLE IF NOT EXISTS {table_ref(BRONZE_EVENTS_QUARANTINE)} (raw_record STRING NOT NULL, _reason STRING NOT NULL, _source_file STRING NOT NULL, _partition_dt DATE, _batch_id STRING NOT NULL, load_timestamp TIMESTAMP NOT NULL) USING DELTA COMMENT 'Raw event records rejected before Bronze' ''')

spark.sql(f'''CREATE TABLE IF NOT EXISTS {table_ref(SILVER_USERS)} (user_id STRING NOT NULL, country STRING, timezone STRING, is_premium BOOLEAN, _row_hash STRING NOT NULL, _batch_id STRING NOT NULL, load_timestamp TIMESTAMP NOT NULL) USING DELTA COMMENT 'Current cleaned user dimension' ''')
spark.sql(f'''CREATE TABLE IF NOT EXISTS {table_ref(SILVER_TRACKS)} (track_id STRING NOT NULL, track_name STRING, artist STRING, genre STRING, release_year INT, duration_sec INT, popularity INT, danceability DOUBLE, energy DOUBLE, valence DOUBLE, tempo_bpm DOUBLE, _row_hash STRING NOT NULL, _batch_id STRING NOT NULL, load_timestamp TIMESTAMP NOT NULL) USING DELTA COMMENT 'Current cleaned track dimension' ''')
spark.sql(f'''CREATE TABLE IF NOT EXISTS {table_ref(SILVER_EVENTS)} (event_id STRING NOT NULL, user_id STRING NOT NULL, track_id STRING NOT NULL, played_at TIMESTAMP NOT NULL, ms_played INT, skipped BOOLEAN, device_type STRING NOT NULL, data_source STRING NOT NULL, is_catalog_matched BOOLEAN NOT NULL, completion_rate DOUBLE, hour_of_day TINYINT NOT NULL, day_of_week TINYINT NOT NULL, is_weekend BOOLEAN NOT NULL, session_id STRING NOT NULL, play_sequence_num INT NOT NULL, dq_flags ARRAY<STRING>, _row_hash STRING NOT NULL, _partition_dt DATE NOT NULL, _batch_id STRING NOT NULL, load_timestamp TIMESTAMP NOT NULL) USING DELTA PARTITIONED BY (_partition_dt) COMMENT 'Current valid listening events' ''')
spark.sql(f'''CREATE TABLE IF NOT EXISTS {table_ref(SILVER_EVENTS_QUARANTINE)} (event_id STRING, user_id STRING, track_id STRING, played_at STRING, ms_played BIGINT, skipped BOOLEAN, device_type STRING, source STRING, _dq_rule STRING NOT NULL, _dq_reason STRING NOT NULL, _quarantined_at TIMESTAMP NOT NULL, _batch_id STRING NOT NULL, load_timestamp TIMESTAMP NOT NULL) USING DELTA COMMENT 'Events rejected by Silver data-quality rules' ''')

spark.sql(f'''CREATE TABLE IF NOT EXISTS {table_ref(INGESTION_LOG)} (_source_file STRING NOT NULL, _partition_dt DATE, sha256 STRING NOT NULL, row_count BIGINT NOT NULL, file_bytes BIGINT NOT NULL, status STRING NOT NULL, reason STRING, bronze_version BIGINT, _batch_id STRING NOT NULL, load_timestamp TIMESTAMP NOT NULL) USING DELTA COMMENT 'Per-file checksum control log' ''')
spark.sql(f'''CREATE TABLE IF NOT EXISTS {table_ref(EXECUTION_LOG)} (log_id STRING NOT NULL, run_id STRING NOT NULL, batch_id STRING NOT NULL, layer STRING NOT NULL, target_table STRING NOT NULL, load_type STRING NOT NULL, parameter STRING NOT NULL, start_time TIMESTAMP NOT NULL, end_time TIMESTAMP NOT NULL, duration_sec DOUBLE, status STRING NOT NULL, rows_read BIGINT, rows_inserted BIGINT, rows_updated BIGINT, rows_deleted BIGINT, rows_quarantined BIGINT, error_message STRING, load_timestamp TIMESTAMP NOT NULL) USING DELTA COMMENT 'Pipeline execution audit log' ''')

for table, name, expression in [(SILVER_EVENTS, 'ms_played_non_negative', 'ms_played IS NULL OR ms_played >= 0'), (SILVER_EVENTS, 'valid_data_source', "data_source IN ('synthetic','synthetic_persona_matched','real_export','real_api')")]:
    try: spark.sql(f'ALTER TABLE {table_ref(table)} ADD CONSTRAINT {name} CHECK ({expression})')
    except Exception as exc:
        if 'already exists' not in str(exc).lower(): raise

print('Phase 2 Bronze, Silver, quarantine, and ops tables are ready.')
